# Session 6: API'er og HTTP-klienter — Byg en klient til din API

## Dagens undervisning

### HTTP-grundbegreber
- Request: metode (GET/POST), URL, headers, body
- Response: statuskode, headers, body (JSON)
- REST API'er: ressourcer, endpoints, autentificering

### httpx — den moderne HTTP-klient

Eksemplerne bruger [GitHub's REST API](https://docs.github.com/en/rest) — gratis, og I har allerede en konto.

In [ ]:
import httpx

# GET — offentlige data, kræver ingen token
response = httpx.get("https://api.github.com/users/octocat")
print(response.status_code)            # 200
data = response.json()
print(data["name"], data["public_repos"])

### Hold hemmeligheder ude af koden

I den her mappe ligger tre filer:

- `.env` — her skriver du din token: `GITHUB_TOKEN=github_pat_...`
- `.env.example` — viser, hvilke variabler `.env` skal indeholde (uden rigtige værdier). Den bliver committet.
- `.gitignore` — sørger for, at `.env` aldrig bliver committet

Lav din token på GitHub: **Settings → Developer settings → Personal access tokens → Fine-grained tokens**. Giv den kun den adgang, den skal bruge: under **Account permissions** sætter du **Gists** til **Read and write**. Den ligger *ikke* under Repository permissions — vælger du forkert, svarer GitHub `404 Not Found`, når du opretter en gist (GitHub svarer 404 i stedet for 403, så den ikke afslører, hvad der findes).

Indsæt din token i `.env`, og kør så cellen nedenfor. Har du ændret `.env`, mens notebooken kører? Så genstart kernen (Kernel → Restart).

In [ ]:
# .env-fil (commit den aldrig!)
# GITHUB_TOKEN=github_pat_...

from dotenv import load_dotenv
import os

load_dotenv()
token = os.environ["GITHUB_TOKEN"]
print("Token indlæst:", bool(token))  # print aldrig selve token — så havner den i notebooken

- `.env` skal i `.gitignore`
- Brug `.env.example` til at dokumentere, hvilke variabler der skal bruges

### GET med token — hvem er jeg?

In [ ]:
response = httpx.get(
    "https://api.github.com/user",
    headers={"Authorization": f"Bearer {token}"},
)
print(response.status_code, response.json()["login"])

# Samme request uden token
response = httpx.get("https://api.github.com/user")
print(response.status_code, response.json()["message"])   # 401 Requires authentication

### POST med JSON-body — opret en hemmelig gist

In [ ]:
response = httpx.post(
    "https://api.github.com/gists",
    headers={"Authorization": f"Bearer {token}"},
    json={
        "description": "Oprettet fra Python",
        "public": False,
        "files": {"hej.txt": {"content": "Hej fra httpx!"}},
    },
)
print(response.status_code)            # 201 Created
response.raise_for_status()            # fejl-status? Så stop her med en tydelig fejl
                                       # 404 her = din token mangler Account permissions → Gists
gist = response.json()
print(gist["html_url"])

Åbn linket og se din gist. Ryd så op igen med en `DELETE`-request:

In [ ]:
response = httpx.delete(
    f"https://api.github.com/gists/{gist['id']}",
    headers={"Authorization": f"Bearer {token}"},
)
print(response.status_code)            # 204 No Content: slettet, ingen body

### Streaming-svar

In [ ]:
# httpbin sender 10 bytes fordelt over 5 sekunder
with httpx.stream("GET", "https://httpbin.org/drip?numbytes=10&duration=5") as response:
    for chunk in response.iter_text():
        print(chunk, end="", flush=True)

- Hvorfor streaming? Oplevet hastighed — data vises, efterhånden som de ankommer, i stedet for først når hele svaret er færdigt. Det er sådan, en AI-chat skriver sit svar ord for ord.
- Server-Sent Events (SSE)-formatet, som AI-chats bruger: `data: {...}\n\n`